# Entrenamiento de YOLOv8 Nano para Detección de Objetos
## Sistema de Inventario Automático

Este notebook entrena un modelo YOLOv8 nano optimizado para detectar objetos del salón de cómputo.

**Objetivos:**
1. Entrenar YOLOv8n con dataset sintético
2. Optimizar el modelo (cuantización)
3. Exportar a TFLite para la aplicación web
4. Evaluar rendimiento

In [ ]:
# Instalación de ultralytics
!pip install ultralytics -q

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from ultralytics import YOLO
import os
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import numpy as np

print("✅ Librerías importadas correctamente")

## Configuración

In [ ]:
# Rutas
YOLO_DIR = "./yolo_dataset"
DATA_YAML = os.path.join(YOLO_DIR, "data.yaml")
MODELS_DIR = "./models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Verificar que existe el archivo data.yaml
if not os.path.exists(DATA_YAML):
    print(f"❌ Error: No se encontró {DATA_YAML}")
    print("   Por favor, ejecuta primero el notebook 01_generar_dataset_sintetico.ipynb")
else:
    print(f"✅ Dataset encontrado: {DATA_YAML}")
    
    # Mostrar configuración
    with open(DATA_YAML, 'r') as f:
        config = yaml.safe_load(f)
    print("\nConfiguración del dataset:")
    print(f"  - Clases: {config['nc']}")
    print(f"  - Nombres: {config['names']}")
    print(f"  - Path: {config['path']}")

## Paso 1: Cargar Modelo Pre-entrenado YOLOv8n

In [ ]:
# Cargar modelo YOLOv8 nano pre-entrenado en COCO
model = YOLO('yolov8n.pt')  # Descarga automáticamente si no existe

print("✅ Modelo YOLOv8 nano cargado")
print(f"\nInformación del modelo:")
print(model.info())

## Paso 2: Entrenar el Modelo

In [ ]:
# Parámetros de entrenamiento
EPOCHS = 100  # Número de épocas
BATCH_SIZE = 16  # Ajustar según memoria GPU disponible
IMG_SIZE = 640  # Tamaño de imagen
DEVICE = 0 if torch.cuda.is_available() else 'cpu'  # GPU si está disponible

print(f"Configuración de entrenamiento:")
print(f"  - Épocas: {EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Tamaño de imagen: {IMG_SIZE}")
print(f"  - Dispositivo: {DEVICE}")
print(f"\n🚀 Iniciando entrenamiento...")
print(f"   Esto puede tomar entre 30 minutos y 2 horas dependiendo del hardware")

In [ ]:
# Entrenar modelo
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project='runs/detect',
    name='inventario_salon',
    patience=15,  # Early stopping si no mejora en 15 épocas
    save=True,
    plots=True,
    # Optimizaciones
    optimizer='AdamW',
    lr0=0.001,  # Learning rate inicial
    lrf=0.01,   # Learning rate final
    momentum=0.937,
    weight_decay=0.0005,
    # Data augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
)

print("\n✅ Entrenamiento completado!")

## Paso 3: Evaluar el Modelo

In [ ]:
# Validar modelo
metrics = model.val()

print("\n📊 Métricas de Validación:")
print(f"  - mAP50: {metrics.box.map50:.4f}")
print(f"  - mAP50-95: {metrics.box.map:.4f}")
print(f"  - Precision: {metrics.box.mp:.4f}")
print(f"  - Recall: {metrics.box.mr:.4f}")

## Paso 4: Visualizar Resultados de Entrenamiento

In [ ]:
# Buscar directorio de resultados
results_dir = Path('runs/detect/inventario_salon')

if results_dir.exists():
    print(f"📁 Resultados guardados en: {results_dir}")
    
    # Mostrar gráficas de entrenamiento
    results_img = results_dir / 'results.png'
    if results_img.exists():
        img = plt.imread(str(results_img))
        plt.figure(figsize=(16, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Métricas de Entrenamiento', fontsize=16, weight='bold')
        plt.tight_layout()
        plt.show()
    
    # Mostrar matriz de confusión
    confusion_matrix_img = results_dir / 'confusion_matrix.png'
    if confusion_matrix_img.exists():
        img = plt.imread(str(confusion_matrix_img))
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Matriz de Confusión', fontsize=16, weight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("⚠️ No se encontró el directorio de resultados")

## Paso 5: Probar Predicciones

In [ ]:
# Cargar el mejor modelo entrenado
best_model_path = results_dir / 'weights' / 'best.pt'

if best_model_path.exists():
    trained_model = YOLO(str(best_model_path))
    print(f"✅ Mejor modelo cargado desde: {best_model_path}")
else:
    trained_model = model
    print("⚠️ Usando modelo actual (no se encontró best.pt)")

In [ ]:
# Probar con imágenes de validación
val_images_dir = Path(YOLO_DIR) / 'val' / 'images'
sample_images = list(val_images_dir.glob('*.jpg'))[:6]  # Primeras 6 imágenes

if sample_images:
    print(f"🔍 Probando modelo con {len(sample_images)} imágenes de validación...")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(sample_images):
        # Realizar predicción
        results = trained_model.predict(str(img_path), conf=0.25, verbose=False)
        
        # Obtener imagen con detecciones dibujadas
        annotated_img = results[0].plot()
        annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
        
        # Mostrar
        axes[idx].imshow(annotated_img)
        
        # Contar detecciones
        num_detections = len(results[0].boxes)
        axes[idx].set_title(f"{img_path.stem}\n{num_detections} objetos detectados", 
                           fontsize=11, weight='bold')
        axes[idx].axis('off')
    
    plt.suptitle('🎯 Predicciones del Modelo Entrenado', fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("❌ No se encontraron imágenes de validación")

## Paso 6: Exportar a TensorFlow Lite (Optimizado)

In [ ]:
print("📦 Exportando modelo a diferentes formatos...\n")

# 1. Exportar a TFLite (Float32) - Sin cuantización
print("1️⃣ Exportando a TFLite (Float32)...")
tflite_float32_path = trained_model.export(
    format='tflite',
    imgsz=IMG_SIZE,
)
print(f"   ✅ Guardado en: {tflite_float32_path}")

# 2. Exportar a TFLite con cuantización INT8
print("\n2️⃣ Exportando a TFLite (INT8 cuantizado)...")
tflite_int8_path = trained_model.export(
    format='tflite',
    imgsz=IMG_SIZE,
    int8=True,  # Cuantización INT8
)
print(f"   ✅ Guardado en: {tflite_int8_path}")

# 3. Exportar a ONNX (útil para debugging)
print("\n3️⃣ Exportando a ONNX...")
onnx_path = trained_model.export(
    format='onnx',
    imgsz=IMG_SIZE,
    simplify=True,
)
print(f"   ✅ Guardado en: {onnx_path}")

print("\n✅ Exportación completada!")

## Paso 7: Comparar Tamaños de Modelos

In [ ]:
import os

def get_file_size(path):
    """Obtiene el tamaño de un archivo en MB"""
    if os.path.exists(path):
        size_bytes = os.path.getsize(path)
        size_mb = size_bytes / (1024 * 1024)
        return size_mb
    return None

print("📊 Comparación de Tamaños de Modelos:")
print("="*60)

models_info = [
    ("PyTorch (.pt)", best_model_path),
    ("TFLite Float32", tflite_float32_path),
    ("TFLite INT8 (Optimizado)", tflite_int8_path),
    ("ONNX", onnx_path),
]

sizes = []
labels = []

for name, path in models_info:
    size = get_file_size(path)
    if size:
        sizes.append(size)
        labels.append(name)
        print(f"{name:>25}: {size:>6.2f} MB")

print("="*60)

# Gráfico comparativo
plt.figure(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = plt.barh(labels, sizes, color=colors[:len(sizes)], alpha=0.8)

# Añadir valores
for bar, size in zip(bars, sizes):
    plt.text(size + 0.1, bar.get_y() + bar.get_height()/2, 
             f'{size:.2f} MB', va='center', fontsize=11, weight='bold')

plt.xlabel('Tamaño (MB)', fontsize=12, weight='bold')
plt.title('📦 Comparación de Tamaños de Modelos', fontsize=14, weight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Recomendación
tflite_int8_size = get_file_size(tflite_int8_path)
print(f"\n💡 RECOMENDACIÓN:")
print(f"   Usar TFLite INT8 ({tflite_int8_size:.2f} MB) para la aplicación web")
print(f"   - Tamaño reducido (~4x más pequeño que Float32)")
print(f"   - Velocidad de inferencia mejorada")
print(f"   - Pérdida mínima de precisión (~1-2%)")

## Paso 8: Copiar Modelo Final a la Carpeta de Modelos

In [ ]:
import shutil

# Copiar modelos a la carpeta de modelos del proyecto
final_model_name = "inventario_yolov8n"

# Copiar PyTorch model
pt_dest = os.path.join(MODELS_DIR, f"{final_model_name}.pt")
shutil.copy2(best_model_path, pt_dest)
print(f"✅ PyTorch model → {pt_dest}")

# Copiar TFLite INT8 (el que usaremos en la web)
tflite_dest = os.path.join(MODELS_DIR, f"{final_model_name}_int8.tflite")
shutil.copy2(tflite_int8_path, tflite_dest)
print(f"✅ TFLite INT8 → {tflite_dest}")

# Copiar TFLite Float32 (alternativa)
tflite_float_dest = os.path.join(MODELS_DIR, f"{final_model_name}_float32.tflite")
shutil.copy2(tflite_float32_path, tflite_float_dest)
print(f"✅ TFLite Float32 → {tflite_float_dest}")

# Crear archivo de labels
labels_path = os.path.join(MODELS_DIR, "labels.txt")
with open(labels_path, 'w') as f:
    for i, label in enumerate(config['names']):
        f.write(f"{i} {label}\n")
print(f"✅ Labels → {labels_path}")

print(f"\n📁 Todos los archivos copiados a: {MODELS_DIR}")

## Paso 9: Crear README con Información del Modelo

In [ ]:
# Crear README con información del modelo
readme_content = f"""# Modelo YOLOv8 Nano - Sistema de Inventario

## 📊 Información del Modelo

### Arquitectura
- **Modelo base**: YOLOv8 Nano
- **Framework**: Ultralytics YOLOv8
- **Tamaño de entrada**: {IMG_SIZE}x{IMG_SIZE}

### Clases Detectadas
{chr(10).join([f'{i}. {label}' for i, label in enumerate(config['names'])])}

### Rendimiento
- **mAP50**: {metrics.box.map50:.4f}
- **mAP50-95**: {metrics.box.map:.4f}
- **Precision**: {metrics.box.mp:.4f}
- **Recall**: {metrics.box.mr:.4f}

### Tamaños de Modelo
- **PyTorch (.pt)**: {get_file_size(best_model_path):.2f} MB
- **TFLite Float32**: {get_file_size(tflite_float32_path):.2f} MB
- **TFLite INT8**: {get_file_size(tflite_int8_path):.2f} MB ⭐ RECOMENDADO

### Dataset de Entrenamiento
- **Imágenes de entrenamiento**: {len(list((Path(YOLO_DIR) / 'train' / 'images').glob('*.jpg')))}
- **Imágenes de validación**: {len(list((Path(YOLO_DIR) / 'val' / 'images').glob('*.jpg')))}
- **Épocas entrenadas**: {EPOCHS}

## 🚀 Uso

### Con Python (PyTorch)
```python
from ultralytics import YOLO

model = YOLO('inventario_yolov8n.pt')
results = model.predict('imagen_salon.jpg')
```

### Con TensorFlow Lite
```python
import tensorflow as tf

interpreter = tf.lite.Interpreter('inventario_yolov8n_int8.tflite')
interpreter.allocate_tensors()
# ... inferencia
```

## 📝 Notas
- Modelo entrenado con dataset sintético generado automáticamente
- Optimizado para detección en tiempo real
- Cuantización INT8 reduce tamaño ~4x con pérdida mínima de precisión
"""

readme_path = os.path.join(MODELS_DIR, "README.md")
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"✅ README creado en: {readme_path}")
print("\n" + "="*60)
print(readme_content)
print("="*60)

## ✅ Entrenamiento Completo

### Siguientes pasos:

1. ✅ Dataset sintético generado
2. ✅ Modelo YOLOv8n entrenado
3. ✅ Modelo exportado a TFLite
4. ⏳ **SIGUIENTE**: Crear aplicación web (`index.html`)

### Archivos generados:
- `models/inventario_yolov8n.pt` - Modelo PyTorch
- `models/inventario_yolov8n_int8.tflite` - Modelo optimizado para web ⭐
- `models/labels.txt` - Lista de clases
- `models/README.md` - Documentación del modelo